In [1]:
import numpy as np
import pandas as pd
import anndata
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import os

import cytovanni

As cytometers vary in how they export their data, how the channels are named etc., we require a CytometerConfig object to set some basic parameters of the data.

# TL;DR

In most cases it should be sufficient to simply set the fluorescence channel names, the (approximate) maximum value at which the channel detectors are still linear, and a name for the cytometer, while using the defaults everywhere else.

In [10]:
cytovanni.io.readfcs_available_channels(os.path.expanduser("~/Cytovanni/data/example_variable-spectra/raw_fcs_original/HD7_202303/2_HD7Tcell20230329/Batch D_1573.fcs"), simple_flowio=False)

array(['FSC-A', 'FSC-H', 'FSC-W', 'SSC-A', 'SSC-H', 'SSC-W', 'BV421-A',
       'BV510-A', 'BV605-A', 'BV650-A', 'BV711-A', 'BV750-A', 'BV786-A',
       'BB515-A', 'BB700-A', 'BUV396-A', 'BUV496-A', 'BUV563-A',
       'BUV615-A', 'BUV661-A', 'BUV737-A', 'BUV805-A', 'APC-A',
       'APC-H7-A', 'BYG584-A', 'PE-CF594-A', 'BYG670-A', 'BYG790-A',
       'APC-R700-A', 'Time'], dtype=object)

In [11]:
channels_fluorescence = ['BB515-A', 'BB700-A',
                         'APC-A', 'APC-H7-A', 'APC-R700-A',
                         'BUV396-A', 'BUV496-A', 'BUV563-A', 'BUV615-A', 'BUV661-A', 'BUV737-A', 'BUV805-A',
                         'BV421-A', 'BV510-A', 'BV605-A', 'BV650-A', 'BV711-A', 'BV750-A', 'BV786-A',
                         'BYG584-A', 'BYG670-A', 'BYG790-A', 'PE-CF594-A']
max_linear_range = 2e5
cytometer = "A3"

In [12]:
cytoconfig = cytovanni.utils.CytometerConfiguration(
                 channels_fluorescence=channels_fluorescence,
                 max_linear_range=max_linear_range,
                 cytometer=cytometer,
                )

In [13]:
cytoconfig

Cytometer configuration for cytometer A3
    Maximum linear intensity: 200000
    Meta channels: ['Time']
    Scatter channels: ['FSC-A', 'FSC-H', 'FSC-W', 'SSC-A', 'SSC-H', 'SSC-W']
        Scatter gating defaults to: ['FSC-A', 'SSC-A']
    Fluorescence channels: ['BB515-A', 'BB700-A', 'APC-A', 'APC-H7-A', 'APC-R700-A', 'BUV396-A', 'BUV496-A', 'BUV563-A', 'BUV615-A', 'BUV661-A', 'BUV737-A', 'BUV805-A', 'BV421-A', 'BV510-A', 'BV605-A', 'BV650-A', 'BV711-A', 'BV750-A', 'BV786-A', 'BYG584-A', 'BYG670-A', 'BYG790-A', 'PE-CF594-A']
    Optional fluorescence channels: ['BB515-H', 'BB515-W', 'BB700-H', 'BB700-W', 'APC-H', 'APC-W', 'APC-H7-H', 'APC-H7-W', 'APC-R700-H', 'APC-R700-W', 'BUV396-H', 'BUV396-W', 'BUV496-H', 'BUV496-W', 'BUV563-H', 'BUV563-W', 'BUV615-H', 'BUV615-W', 'BUV661-H', 'BUV661-W', 'BUV737-H', 'BUV737-W', 'BUV805-H', 'BUV805-W', 'BV421-H', 'BV421-W', 'BV510-H', 'BV510-W', 'BV605-H', 'BV605-W', 'BV650-H', 'BV650-W', 'BV711-H', 'BV711-W', 'BV750-H', 'BV750-W', 'BV786-H', 'BV7

If this is used repeatedly, it may be convenient to create a new class:

In [14]:
class CytometerConfiguration_A3(cytovanni.utils.CytometerConfiguration):
    """ Cytometer configuration for our A3.
    """
    def __init__(self):
        channels_fluorescence = ['BB515-A', 'BB700-A',
                                 'APC-A', 'APC-H7-A', 'APC-R700-A',
                                 'BUV396-A', 'BUV496-A', 'BUV563-A', 'BUV615-A', 'BUV661-A', 'BUV737-A', 'BUV805-A',
                                 'BV421-A', 'BV510-A', 'BV605-A', 'BV650-A', 'BV711-A', 'BV750-A', 'BV786-A',
                                 'BYG584-A', 'BYG670-A', 'BYG790-A', 'PE-CF594-A']
        
        super().__init__(
            channels_fluorescence=channels_fluorescence,
            cytometer="A3",
            max_linear_range=2e5
        )

In [15]:
cytoconfig = CytometerConfiguration_A3()
cytoconfig

Cytometer configuration for cytometer A3
    Maximum linear intensity: 200000
    Meta channels: ['Time']
    Scatter channels: ['FSC-A', 'FSC-H', 'FSC-W', 'SSC-A', 'SSC-H', 'SSC-W']
        Scatter gating defaults to: ['FSC-A', 'SSC-A']
    Fluorescence channels: ['BB515-A', 'BB700-A', 'APC-A', 'APC-H7-A', 'APC-R700-A', 'BUV396-A', 'BUV496-A', 'BUV563-A', 'BUV615-A', 'BUV661-A', 'BUV737-A', 'BUV805-A', 'BV421-A', 'BV510-A', 'BV605-A', 'BV650-A', 'BV711-A', 'BV750-A', 'BV786-A', 'BYG584-A', 'BYG670-A', 'BYG790-A', 'PE-CF594-A']
    Optional fluorescence channels: ['BB515-H', 'BB515-W', 'BB700-H', 'BB700-W', 'APC-H', 'APC-W', 'APC-H7-H', 'APC-H7-W', 'APC-R700-H', 'APC-R700-W', 'BUV396-H', 'BUV396-W', 'BUV496-H', 'BUV496-W', 'BUV563-H', 'BUV563-W', 'BUV615-H', 'BUV615-W', 'BUV661-H', 'BUV661-W', 'BUV737-H', 'BUV737-W', 'BUV805-H', 'BUV805-W', 'BV421-H', 'BV421-W', 'BV510-H', 'BV510-W', 'BV605-H', 'BV605-W', 'BV650-H', 'BV650-W', 'BV711-H', 'BV711-W', 'BV750-H', 'BV750-W', 'BV786-H', 'BV7

The configuration can alternatively be written into a json and later loaded again:

In [16]:
cytoconfig.write_json("cytoconfig_A3.json")

In [17]:
cytovanni.utils.CytometerConfiguration.load_json("cytoconfig_A3.json")

Cytometer configuration for cytometer A3
    Maximum linear intensity: 200000
    Meta channels: ['Time']
    Scatter channels: ['FSC-A', 'FSC-H', 'FSC-W', 'SSC-A', 'SSC-H', 'SSC-W']
        Scatter gating defaults to: ['FSC-A', 'SSC-A']
    Fluorescence channels: ['BB515-A', 'BB700-A', 'APC-A', 'APC-H7-A', 'APC-R700-A', 'BUV396-A', 'BUV496-A', 'BUV563-A', 'BUV615-A', 'BUV661-A', 'BUV737-A', 'BUV805-A', 'BV421-A', 'BV510-A', 'BV605-A', 'BV650-A', 'BV711-A', 'BV750-A', 'BV786-A', 'BYG584-A', 'BYG670-A', 'BYG790-A', 'PE-CF594-A']
    Optional fluorescence channels: ['BB515-H', 'BB515-W', 'BB700-H', 'BB700-W', 'APC-H', 'APC-W', 'APC-H7-H', 'APC-H7-W', 'APC-R700-H', 'APC-R700-W', 'BUV396-H', 'BUV396-W', 'BUV496-H', 'BUV496-W', 'BUV563-H', 'BUV563-W', 'BUV615-H', 'BUV615-W', 'BUV661-H', 'BUV661-W', 'BUV737-H', 'BUV737-W', 'BUV805-H', 'BUV805-W', 'BV421-H', 'BV421-W', 'BV510-H', 'BV510-W', 'BV605-H', 'BV605-W', 'BV650-H', 'BV650-W', 'BV711-H', 'BV711-W', 'BV750-H', 'BV750-W', 'BV786-H', 'BV7

# Full

In [ ]:
cytovanni.utils.CytometerConfiguration(channels_fluorescence,
                 channels_scatter=["FSC-A","FSC-H","FSC-W",
                                   "SSC-A","SSC-H","SSC-W"],
                 default_scatter_gate=["FSC-A", "SSC-A"],
                 channels_meta=["Time"],
                 max_linear_range=2e5,
                 cytometer="default",
                 try_adding_HW=False,
                 fct_hash_settings=cytovanni.utils.config._default_hash_settings,
                )

First, we require setting all the scatter and fluorescence channels that should be loaded from the .fcs files. They need to be chosen from the available channels:

In [20]:
cytovanni.io.readfcs_available_channels(os.path.expanduser("~/Cytovanni/data/example_variable-spectra/raw_fcs_original/HD7_202303/2_HD7Tcell20230329/Batch D_1573.fcs"), simple_flowio=False)

array(['FSC-A', 'FSC-H', 'FSC-W', 'SSC-A', 'SSC-H', 'SSC-W', 'BV421-A',
       'BV510-A', 'BV605-A', 'BV650-A', 'BV711-A', 'BV750-A', 'BV786-A',
       'BB515-A', 'BB700-A', 'BUV396-A', 'BUV496-A', 'BUV563-A',
       'BUV615-A', 'BUV661-A', 'BUV737-A', 'BUV805-A', 'APC-A',
       'APC-H7-A', 'BYG584-A', 'PE-CF594-A', 'BYG670-A', 'BYG790-A',
       'APC-R700-A', 'Time'], dtype=object)

The fluorescence channels vary widely between intruments, so they always need to be adjusted.

In [21]:
channels_fluorescence = ['BB515-A', 'BB700-A',
                         'APC-A', 'APC-H7-A', 'APC-R700-A',
                         'BUV396-A', 'BUV496-A', 'BUV563-A', 'BUV615-A', 'BUV661-A', 'BUV737-A', 'BUV805-A',
                         'BV421-A', 'BV510-A', 'BV605-A', 'BV650-A', 'BV711-A', 'BV750-A', 'BV786-A',
                         'BYG584-A', 'BYG670-A', 'BYG790-A', 'PE-CF594-A']

The scatter channels are usually the same across instruments, so this is also set as the default.

In [22]:
channels_scatter = ["FSC-A","FSC-H","FSC-W",
                    "SSC-A","SSC-H","SSC-W"]

Similarly, most cytometers only export the time of recording as additional per event metadata.

In [23]:
channels_meta = ["Time"]

It is usually possible to also export the height -H and width -W for every fluorescence channel. If they should be automatically loaded where available, set:

In [24]:
try_adding_HW = True

Some estimations also require a rough value for where the linear range of the fluorescence detectors ends:

In [25]:
max_linear_range = 2e5

It is also possible to set a name for the cytometer.

In [26]:
cytometer = "A3"

While our bead gating functions can take the two channels that should be used as an argument, this can also be set as a default here.

In [27]:
default_scatter_gate = ["FSC-A", "SSC-A"]

To ensure that the cytometer settings didn't accidentally get mixed up within a single batch, we also calculate a hash of the channel settings for every sample. The default uses the read gains and voltages for every loaded channel; this should usually be fine but can be customized if necessary. Note that this function is the only part of the cytometer settings that will be always be overwritten by the default if the cytometer configuration is read from an exported file.

In [28]:
fct_hash_settings = cytovanni.utils.config._default_hash_settings

All of this can then be combined into a single object:

In [29]:
cytoconfig = cytovanni.utils.CytometerConfiguration(
                 channels_fluorescence=channels_fluorescence,
                 channels_scatter=channels_scatter,
                 default_scatter_gate=default_scatter_gate,
                 channels_meta=channels_meta,
                 max_linear_range=max_linear_range,
                 cytometer=cytometer,
                 try_adding_HW=try_adding_HW,
                 fct_hash_settings=fct_hash_settings,
                )

In [30]:
cytoconfig

Cytometer configuration for cytometer A3
    Maximum linear intensity: 200000
    Meta channels: ['Time']
    Scatter channels: ['FSC-A', 'FSC-H', 'FSC-W', 'SSC-A', 'SSC-H', 'SSC-W']
        Scatter gating defaults to: ['FSC-A', 'SSC-A']
    Fluorescence channels: ['BB515-A', 'BB700-A', 'APC-A', 'APC-H7-A', 'APC-R700-A', 'BUV396-A', 'BUV496-A', 'BUV563-A', 'BUV615-A', 'BUV661-A', 'BUV737-A', 'BUV805-A', 'BV421-A', 'BV510-A', 'BV605-A', 'BV650-A', 'BV711-A', 'BV750-A', 'BV786-A', 'BYG584-A', 'BYG670-A', 'BYG790-A', 'PE-CF594-A']
    Optional fluorescence channels: ['BB515-H', 'BB515-W', 'BB700-H', 'BB700-W', 'APC-H', 'APC-W', 'APC-H7-H', 'APC-H7-W', 'APC-R700-H', 'APC-R700-W', 'BUV396-H', 'BUV396-W', 'BUV496-H', 'BUV496-W', 'BUV563-H', 'BUV563-W', 'BUV615-H', 'BUV615-W', 'BUV661-H', 'BUV661-W', 'BUV737-H', 'BUV737-W', 'BUV805-H', 'BUV805-W', 'BV421-H', 'BV421-W', 'BV510-H', 'BV510-W', 'BV605-H', 'BV605-W', 'BV650-H', 'BV650-W', 'BV711-H', 'BV711-W', 'BV750-H', 'BV750-W', 'BV786-H', 'BV7